In [1]:
from pyspark.sql import functions as F
from minio_config import minio_path


# ==================================================
# LOAD GOLD DIMENSIONS
# ==================================================

dim_property_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_property"
    )
)

dim_building_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_building"
    )
)


# ==================================================
# LOAD NYC 311 SILVER
# ==================================================

nyc311_df = spark.read.parquet(
    minio_path(
        "silver/nyc_311"
    )
)


# ==================================================
# LOAD EXISTING 311 RESOLUTION RESULTS
# ==================================================

unique_bbl_resolved_df = spark.read.parquet(
    minio_path(
        "gold/building_risk/intermediate/"
        "311_unique_bbl_resolved"
    )
)

multi_bbl_address_resolved_df = spark.read.parquet(
    minio_path(
        "gold/building_risk/intermediate/"
        "311_multi_bbl_address_resolved"
    )
)

global_address_resolved_df = spark.read.parquet(
    minio_path(
        "gold/building_risk/intermediate/"
        "311_global_address_resolved"
    )
)

print("311 Gold sources loaded successfully")

ModuleNotFoundError: No module named 'minio_config'

In [2]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# ==================================================
# PROJECT CONFIG
# ==================================================

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)


# ==================================================
# IMPORT PROJECT HELPERS
# ==================================================

from minio_config import configure_minio, minio_path


# ==================================================
# CREATE / GET SPARK SESSION
# ==================================================

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Gold Fact 311 Event")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .getOrCreate()
)

configure_minio(spark)

spark.sparkContext.setLogLevel("WARN")


# ==================================================
# TEST
# ==================================================

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("MinIO helper loaded successfully")
print("Test:", spark.range(1).count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/07 17:06:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/07 17:06:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/07 17:06:58 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark version: 3.4.0
Master: local[2]
MinIO helper loaded successfully
Test: 1


In [3]:
# ==================================================
# LOAD GOLD DIMENSIONS
# ==================================================

dim_property_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_property"
    )
)

dim_building_df = spark.read.parquet(
    minio_path(
        "gold/data_model/dim_building"
    )
)


# ==================================================
# LOAD NYC 311 SILVER
# ==================================================

nyc311_df = spark.read.parquet(
    minio_path(
        "silver/nyc_311"
    )
)


# ==================================================
# LOAD EXISTING 311 RESOLUTION RESULTS
# ==================================================

unique_bbl_resolved_df = spark.read.parquet(
    minio_path(
        "gold/building_risk/intermediate/"
        "311_unique_bbl_resolved"
    )
)

multi_bbl_address_resolved_df = spark.read.parquet(
    minio_path(
        "gold/building_risk/intermediate/"
        "311_multi_bbl_address_resolved"
    )
)

global_address_resolved_df = spark.read.parquet(
    minio_path(
        "gold/building_risk/intermediate/"
        "311_global_address_resolved"
    )
)


print("311 Gold sources loaded successfully")

26/09/07 17:07:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


311 Gold sources loaded successfully


In [4]:
# ==================================================
# BUILD UNIFIED 311 RESOLUTION MAP
# ==================================================

map_unique_bbl = (
    unique_bbl_resolved_df
    .select(
        "unique_key",
        "resolved_bin",
        "match_method",
        "match_confidence",
        "resolution_status"
    )
)


map_multi_bbl_address = (
    multi_bbl_address_resolved_df
    .select(
        "unique_key",
        "resolved_bin",
        "match_method",
        "match_confidence",
        "resolution_status"
    )
)


map_global_address = (
    global_address_resolved_df
    .select(
        "unique_key",
        "resolved_bin",
        "match_method",
        "match_confidence",
        "resolution_status"
    )
)


nyc311_resolved_map = (
    map_unique_bbl
    .unionByName(map_multi_bbl_address)
    .unionByName(map_global_address)
)


print(
    "Resolved mapping rows:",
    nyc311_resolved_map.count()
)

print(
    "Distinct resolved 311 events:",
    nyc311_resolved_map
    .select("unique_key")
    .distinct()
    .count()
)

(
    nyc311_resolved_map
    .groupBy("match_method")
    .count()
    .orderBy("match_method")
    .show(truncate=False)
)

Resolved mapping rows: 839459


Distinct resolved 311 events: 839459
+---------------------+------+
|match_method         |count |
+---------------------+------+
|BBL_EXACT_ADDRESS    |95658 |
|EXACT_ADDRESS_BOROUGH|446   |
|UNIQUE_BBL           |743355|
+---------------------+------+



In [5]:
# ==================================================
# 311 RESOLVED EVENTS -> DIM_BUILDING
# ==================================================

building_lookup = (
    dim_building_df

    .select(
        F.col("bin")
        .cast("string")
        .alias("resolved_bin"),

        "building_id",

        F.col("property_id")
        .alias("building_property_id")
    )

    .dropDuplicates(
        ["resolved_bin"]
    )
)


nyc311_resolved_identity = (
    nyc311_resolved_map

    .withColumn(
        "resolved_bin",
        F.col("resolved_bin").cast("string")
    )

    .join(
        building_lookup,
        on="resolved_bin",
        how="left"
    )
)

In [6]:
# ==================================================
# 311 RESOLVED EVENTS -> DIM_BUILDING
# ==================================================

building_lookup = (
    dim_building_df

    .select(
        F.col("bin")
        .cast("string")
        .alias("resolved_bin"),

        "building_id",

        F.col("property_id")
        .alias("building_property_id")
    )

    .dropDuplicates(
        ["resolved_bin"]
    )
)


nyc311_resolved_identity = (
    nyc311_resolved_map

    .withColumn(
        "resolved_bin",
        F.col("resolved_bin").cast("string")
    )

    .join(
        building_lookup,
        on="resolved_bin",
        how="left"
    )
)

In [7]:
print(
    "Resolved 311 events:",
    nyc311_resolved_identity.count()
)

print(
    "With building_id:",
    nyc311_resolved_identity
    .filter(
        F.col("building_id").isNotNull()
    )
    .count()
)

print(
    "Without building_id:",
    nyc311_resolved_identity
    .filter(
        F.col("building_id").isNull()
    )
    .count()
)

print(
    "With building property_id:",
    nyc311_resolved_identity
    .filter(
        F.col("building_property_id").isNotNull()
    )
    .count()
)

print(
    "Without building property_id:",
    nyc311_resolved_identity
    .filter(
        F.col("building_property_id").isNull()
    )
    .count()
)

Resolved 311 events: 839459


With building_id: 839459
Without building_id: 0
With building property_id: 838457
Without building property_id: 1002


In [8]:
# ==================================================
# 311 SOURCE BBL -> DIM_PROPERTY
# ==================================================

property_lookup = (
    dim_property_df
    .select(
        F.col("bbl")
        .cast("string")
        .alias("source_bbl"),

        F.col("property_id")
        .alias("source_property_id")
    )
    .dropDuplicates(["source_bbl"])
)


nyc311_with_source_property = (
    nyc311_df

    .select(
        "unique_key",
        F.col("bbl")
        .cast("string")
        .alias("source_bbl")
    )

    .join(
        property_lookup,
        on="source_bbl",
        how="left"
    )
)

In [9]:
# ==================================================
# ADD SOURCE PROPERTY TO RESOLVED 311 EVENTS
# ==================================================

nyc311_resolved_identity_v2 = (
    nyc311_resolved_identity

    .join(
        nyc311_with_source_property,
        on="unique_key",
        how="left"
    )

    # canonical property of the resolved building
    .withColumn(
        "property_id",
        F.col("building_property_id")
    )
)

In [10]:
unresolved_building_property = (
    nyc311_resolved_identity_v2
    .filter(
        F.col("property_id").isNull()
    )
)

print(
    "311 events without canonical property_id:",
    unresolved_building_property.count()
)

print(
    "Of them with valid source_property_id:",
    unresolved_building_property
    .filter(
        F.col("source_property_id").isNotNull()
    )
    .count()
)

print(
    "Of them without source_property_id:",
    unresolved_building_property
    .filter(
        F.col("source_property_id").isNull()
    )
    .count()
)

311 events without canonical property_id: 1002


Of them with valid source_property_id: 55


Of them without source_property_id: 947


In [11]:
# ==================================================
# LOAD REMAINING UNRESOLVED 311 EVENTS
# ==================================================

unresolved_311_df = spark.read.parquet(
    minio_path(
        "gold/building_risk/intermediate/"
        "311_unresolved_after_address"
    )
)


# ==================================================
# LOAD EXISTING GEO PROFILE
# ==================================================

geo_profile_df = spark.read.parquet(
    minio_path(
        "gold/building_risk/intermediate/"
        "311_geo_profile"
    )
)


print(
    "Unresolved 311 events:",
    unresolved_311_df.count()
)

print(
    "Geo profile rows:",
    geo_profile_df.count()
)

Unresolved 311 events: 45847
Geo profile rows: 44837


In [12]:
print("GEO PROFILE SCHEMA")
geo_profile_df.printSchema()

GEO PROFILE SCHEMA
root
 |-- unique_key: string (nullable = true)
 |-- candidate_count_100m: long (nullable = true)
 |-- nearest_bin: string (nullable = true)
 |-- nearest_distance_m: double (nullable = true)
 |-- second_distance_m: double (nullable = true)



In [13]:
geo_profile_df.show(
    10,
    truncate=False
)

+----------+--------------------+-----------+------------------+------------------+
|unique_key|candidate_count_100m|nearest_bin|nearest_distance_m|second_distance_m |
+----------+--------------------+-----------+------------------+------------------+
|65963184  |3                   |2046804    |38.447276183571745|71.78301671444585 |
|65963191  |33                  |2094321    |9.656666674551941 |45.59468498568239 |
|65964405  |22                  |3076282    |26.870396992674063|34.734624903455604|
|65964414  |32                  |3016259    |1.0890431836330274|1.3396392005515019|
|65964433  |15                  |3330277    |7.8987318037521375|22.843892741629706|
|65964440  |15                  |3330277    |7.8987318037521375|22.843892741629706|
|65965564  |17                  |2005845    |18.839758853720827|19.098508092270414|
|65965603  |15                  |3330277    |7.8987318037521375|22.843892741629706|
|65965640  |47                  |3078174    |5.247283329477307 |15.675468022

In [14]:
# ==================================================
# CONSERVATIVE GEO RESOLUTION
# ==================================================

geo_resolution_df = (
    geo_profile_df

    .withColumn(
        "distance_gap_m",
        F.col("second_distance_m") -
        F.col("nearest_distance_m")
    )

    .withColumn(
        "match_method",
        F.when(
            (F.col("candidate_count_100m") == 1) &
            (F.col("nearest_distance_m") <= 25),
            F.lit("GEO_UNIQUE_25M")
        )
        .when(
            (F.col("nearest_distance_m") <= 10) &
            (F.col("distance_gap_m") >= 20),
            F.lit("GEO_DOMINANT_10M")
        )
    )

    .filter(
        F.col("match_method").isNotNull()
    )

    .withColumn(
        "resolved_bin",
        F.col("nearest_bin")
    )

    .withColumn(
        "match_confidence",
        F.lit("HIGH")
    )

    .withColumn(
        "resolution_status",
        F.lit("AUTO_RESOLVED")
    )

    .select(
        "unique_key",
        "resolved_bin",
        "match_method",
        "match_confidence",
        "resolution_status",
        "candidate_count_100m",
        "nearest_distance_m",
        "second_distance_m",
        "distance_gap_m"
    )
)

In [15]:
print(
    "GEO auto-resolved events:",
    geo_resolution_df.count()
)

(
    geo_resolution_df
    .groupBy("match_method")
    .count()
    .show(truncate=False)
)

GEO auto-resolved events: 3739
+----------------+-----+
|match_method    |count|
+----------------+-----+
|GEO_DOMINANT_10M|3543 |
|GEO_UNIQUE_25M  |196  |
+----------------+-----+



In [16]:
# ==================================================
# SAVE CONSERVATIVE GEO RESOLUTION
# ==================================================

GEO_RESOLUTION_PATH = minio_path(
    "gold/data_model/intermediate/311_geo_resolved"
)

(
    geo_resolution_df
    .write
    .mode("overwrite")
    .parquet(GEO_RESOLUTION_PATH)
)

print("311 GEO resolution saved successfully")
print("Path:", GEO_RESOLUTION_PATH)

311 GEO resolution saved successfully
Path: s3a://nyc-building-risk/gold/data_model/intermediate/311_geo_resolved


In [17]:
# ==================================================
# ADD GEO TO EXISTING 311 RESOLUTION MAP
# ==================================================

geo_map = (
    geo_resolution_df
    .select(
        "unique_key",
        "resolved_bin",
        "match_method",
        "match_confidence",
        "resolution_status"
    )
)


nyc311_all_resolved_map = (
    nyc311_resolved_map
    .unionByName(
        geo_map
    )
)

In [18]:
print(
    "All resolved mapping rows:",
    nyc311_all_resolved_map.count()
)

print(
    "Distinct resolved events:",
    nyc311_all_resolved_map
    .select("unique_key")
    .distinct()
    .count()
)

(
    nyc311_all_resolved_map
    .groupBy("match_method")
    .count()
    .orderBy("match_method")
    .show(truncate=False)
)

All resolved mapping rows: 843198


Distinct resolved events: 843198
+---------------------+------+
|match_method         |count |
+---------------------+------+
|BBL_EXACT_ADDRESS    |95658 |
|EXACT_ADDRESS_BOROUGH|446   |
|GEO_DOMINANT_10M     |3543  |
|GEO_UNIQUE_25M       |196   |
|UNIQUE_BBL           |743355|
+---------------------+------+



In [19]:
print("NYC 311 SILVER SCHEMA")
nyc311_df.printSchema()

NYC 311 SILVER SCHEMA
root
 |-- unique_key: string (nullable = true)
 |-- created_date: timestamp (nullable = true)
 |-- closed_date: timestamp (nullable = true)
 |-- resolution_action_updated_date: timestamp (nullable = true)
 |-- agency: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- complaint_type: string (nullable = true)
 |-- descriptor: string (nullable = true)
 |-- descriptor_2: string (nullable = true)
 |-- status: string (nullable = true)
 |-- incident_address: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- incident_zip: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- city: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- community_board: string (nullable = true)
 |-- council_district: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- open_data_channel_type: string (nullable = true)
 |

In [20]:
# ==================================================
# BUILD FACT_311_EVENT - STAGING
# ==================================================

# 311 source data
nyc311_base = (
    nyc311_df
    .select(
        "unique_key",
        "created_date",
        "closed_date",
        "resolution_action_updated_date",
        "agency",
        "agency_name",
        "complaint_type",
        "descriptor",
        "descriptor_2",
        "status",
        "incident_address",
        "street_name",
        "incident_zip",
        "borough",
        "city",

        F.col("bbl")
        .cast("string")
        .alias("source_bbl"),

        "latitude",
        "longitude",
        "community_board",
        "council_district",
        "location_type",
        "open_data_channel_type",
        "created_day",
        "created_year",
        "created_month"
    )
)


# Building lookup
building_lookup = (
    dim_building_df
    .select(
        F.col("bin")
        .cast("string")
        .alias("resolved_bin"),

        "building_id",

        F.col("property_id")
        .alias("building_property_id")
    )
    .dropDuplicates(["resolved_bin"])
)


# Property lookup from original 311 BBL
property_lookup = (
    dim_property_df
    .select(
        F.col("bbl")
        .cast("string")
        .alias("source_bbl"),

        F.col("property_id")
        .alias("source_property_id")
    )
    .dropDuplicates(["source_bbl"])
)


# Complete resolution map
resolution_map = (
    nyc311_all_resolved_map
    .withColumn(
        "resolved_bin",
        F.col("resolved_bin").cast("string")
    )
)


# Build staging fact
fact_311_stage = (
    nyc311_base

    .join(
        resolution_map,
        on="unique_key",
        how="left"
    )

    .join(
        building_lookup,
        on="resolved_bin",
        how="left"
    )

    .join(
        property_lookup,
        on="source_bbl",
        how="left"
    )

    .withColumn(
        "building_resolution_status",
        F.when(
            F.col("building_id").isNotNull(),
            F.lit("RESOLVED")
        ).otherwise(
            F.lit("UNRESOLVED")
        )
    )
)

In [21]:
print(
    "Total 311 events:",
    fact_311_stage.count()
)

print(
    "Distinct unique_key:",
    fact_311_stage
    .select("unique_key")
    .distinct()
    .count()
)

print(
    "With building_id:",
    fact_311_stage
    .filter(F.col("building_id").isNotNull())
    .count()
)

print(
    "Without building_id:",
    fact_311_stage
    .filter(F.col("building_id").isNull())
    .count()
)

Total 311 events: 885306


Distinct unique_key: 885306


With building_id: 843198


Without building_id: 42108


In [22]:
both_property = (
    F.col("building_property_id").isNotNull()
    & F.col("source_property_id").isNotNull()
)

print(
    "Both Property IDs available:",
    fact_311_stage
    .filter(both_property)
    .count()
)

print(
    "Same Property:",
    fact_311_stage
    .filter(
        both_property
        & (
            F.col("building_property_id")
            == F.col("source_property_id")
        )
    )
    .count()
)

print(
    "Different Property:",
    fact_311_stage
    .filter(
        both_property
        & (
            F.col("building_property_id")
            != F.col("source_property_id")
        )
    )
    .count()
)

print(
    "Building Property only:",
    fact_311_stage
    .filter(
        F.col("building_property_id").isNotNull()
        & F.col("source_property_id").isNull()
    )
    .count()
)

print(
    "Source Property only:",
    fact_311_stage
    .filter(
        F.col("building_property_id").isNull()
        & F.col("source_property_id").isNotNull()
    )
    .count()
)

print(
    "No Property:",
    fact_311_stage
    .filter(
        F.col("building_property_id").isNull()
        & F.col("source_property_id").isNull()
    )
    .count()
)

Both Property IDs available: 840746


Same Property: 835201


Different Property: 5545


Building Property only: 1438


Source Property only: 39717


No Property: 3405


In [23]:
# ==================================================
# FINAL PROPERTY RESOLUTION FOR 311
# ==================================================

fact_311_final = (
    fact_311_stage

    # --------------------------------------------------
    # Canonical Property FK
    # --------------------------------------------------
    .withColumn(
        "property_id",
        F.when(
            F.col("building_property_id").isNotNull(),
            F.col("building_property_id")
        )
        .when(
            F.col("building_id").isNull()
            & F.col("source_property_id").isNotNull(),
            F.col("source_property_id")
        )
    )

    # --------------------------------------------------
    # How was Property resolved?
    # --------------------------------------------------
    .withColumn(
        "property_resolution_method",
        F.when(
            F.col("building_property_id").isNotNull(),
            F.lit("BUILDING_CANONICAL")
        )
        .when(
            F.col("building_id").isNull()
            & F.col("source_property_id").isNotNull(),
            F.lit("SOURCE_BBL")
        )
        .otherwise(
            F.lit("UNRESOLVED")
        )
    )

    # --------------------------------------------------
    # Source BBL vs canonical Building Property conflict
    # --------------------------------------------------
    .withColumn(
        "property_conflict_flag",
        F.when(
            F.col("building_property_id").isNotNull()
            & F.col("source_property_id").isNotNull()
            & (
                F.col("building_property_id")
                != F.col("source_property_id")
            ),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    # Standard final Building status
    .withColumn(
        "building_resolution_status",
        F.when(
            F.col("building_id").isNotNull(),
            F.lit("RESOLVED")
        ).otherwise(
            F.lit("UNRESOLVED")
        )
    )
)

In [24]:
print(
    "Total fact rows:",
    fact_311_final.count()
)

print(
    "With building_id:",
    fact_311_final
    .filter(F.col("building_id").isNotNull())
    .count()
)

print(
    "With property_id:",
    fact_311_final
    .filter(F.col("property_id").isNotNull())
    .count()
)

print(
    "Without property_id:",
    fact_311_final
    .filter(F.col("property_id").isNull())
    .count()
)

print(
    "Property conflicts:",
    fact_311_final
    .filter(F.col("property_conflict_flag") == 1)
    .count()
)

(
    fact_311_final
    .groupBy("property_resolution_method")
    .count()
    .show(truncate=False)
)

Total fact rows: 885306


With building_id: 843198


With property_id: 881836


Without property_id: 3470


Property conflicts: 5545


+--------------------------+------+
|property_resolution_method|count |
+--------------------------+------+
|UNRESOLVED                |3470  |
|SOURCE_BBL                |39652 |
|BUILDING_CANONICAL        |842184|
+--------------------------+------+



In [25]:
# ==================================================
# FINAL FACT_311_EVENT STRUCTURE
# Grain: 1 row = 1 NYC 311 event
# ==================================================

fact_311_event = (
    fact_311_final

    .withColumn(
        "event_id",
        F.concat(
            F.lit("311:"),
            F.col("unique_key")
        )
    )

    .select(
        # -----------------------------
        # Keys
        # -----------------------------
        "event_id",
        "unique_key",

        "property_id",
        "building_id",

        # -----------------------------
        # Identity / Audit
        # -----------------------------
        "resolved_bin",
        "source_bbl",
        "source_property_id",
        "building_property_id",

        "match_method",
        "match_confidence",
        "resolution_status",

        "building_resolution_status",
        "property_resolution_method",
        "property_conflict_flag",

        # -----------------------------
        # Event
        # -----------------------------
        "created_date",
        "closed_date",
        "resolution_action_updated_date",

        "agency",
        "agency_name",
        "complaint_type",
        "descriptor",
        "descriptor_2",
        "status",

        # -----------------------------
        # Location from source
        # -----------------------------
        "incident_address",
        "street_name",
        "incident_zip",
        "borough",
        "city",
        "latitude",
        "longitude",

        "community_board",
        "council_district",
        "location_type",
        "open_data_channel_type",

        # -----------------------------
        # Calendar helpers
        # -----------------------------
        "created_day",
        "created_year",
        "created_month"
    )
)

In [26]:
print(
    "Fact rows:",
    fact_311_event.count()
)

print(
    "Distinct event_id:",
    fact_311_event
    .select("event_id")
    .distinct()
    .count()
)

print(
    "With building_id:",
    fact_311_event
    .filter(F.col("building_id").isNotNull())
    .count()
)

print(
    "With property_id:",
    fact_311_event
    .filter(F.col("property_id").isNotNull())
    .count()
)

Fact rows: 885306


Distinct event_id: 885306


With building_id: 843198


With property_id: 881836


In [27]:
# ==================================================
# SAVE FACT_311_EVENT
# ==================================================

FACT_311_PATH = minio_path(
    "gold/data_model/fact_311_event"
)

(
    fact_311_event
    .write
    .mode("overwrite")
    .parquet(FACT_311_PATH)
)

print("fact_311_event saved successfully")
print("Path:", FACT_311_PATH)

26/09/07 17:42:07 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


fact_311_event saved successfully
Path: s3a://nyc-building-risk/gold/data_model/fact_311_event


In [28]:
# ==================================================
# SHOW GOLD DIRECTORY STRUCTURE
# ==================================================

jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()

gold_path = jvm.org.apache.hadoop.fs.Path(
    minio_path("gold")
)

fs = gold_path.getFileSystem(hadoop_conf)


def show_dirs(path, level=0, max_depth=3):
    if level > max_depth:
        return

    statuses = fs.listStatus(path)

    for status in statuses:
        if status.isDirectory():
            p = status.getPath()

            print(
                "    " * level +
                p.getName() +
                "/"
            )

            show_dirs(
                p,
                level + 1,
                max_depth
            )


show_dirs(gold_path)

building_identity/
    building_master/
    elasticsearch_resolver/
    final/
    intermediate/
        bbl_history_recovered/
        building_observations/
        direct_bbl_unmatched/
        exact_address_ambiguous/
        exact_address_recovered/
        unresolved_after_bbl_history/
    unresolved/
building_risk/
    intermediate/
        311_geo_profile/
        311_global_address_resolved/
        311_multi_bbl_address_resolved/
        311_unique_bbl_resolved/
        311_unresolved_after_address/
        311_unresolved_after_bbl/
        311_unresolved_after_multi_bbl_address/
        bbl_to_bin_reference/
        dob_features/
        hpd_features/
data_model/
    dim_building/
    dim_property/
    fact_311_event/
    intermediate/
        311_geo_resolved/
